In [69]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import operator

In [70]:
generator_llm = ChatOpenAI(model = 'gpt-4o-mini')
evaluator_llm = ChatOpenAI(model = 'gpt-4o-mini')
optimizer_llm = ChatOpenAI(model = 'gpt-4o-mini')

In [71]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final Evaluation")
    feedback: str = Field(..., description="feedback for the tweet.")

In [72]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [73]:
#state
from typing import Annotated


class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str],operator.add]
    iteration: int #number of loops between evaluaton and optimizer
    max_iteration: int

In [74]:
def generate_tweet(state: TweetState):
    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
- This is version {state['iteration'] + 1}.
""")
    ]

    response = generator_llm.invoke(messages).content
    return {'tweet': response, 'tweet_history': [response]}


In [75]:
def evaluate_tweet(state: TweetState):
      # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    
    response = structured_evaluator_llm.invoke(messages)
    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [76]:
def optimize_tweet(state: TweetState):
     messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
     
     response = optimizer_llm.invoke(messages).content
     iteration = state['iteration']+1
     return {'tweet': response , 'iteration': iteration,'tweet_history': [response]}


In [77]:
def route_evaluation(state: TweetState):
    if(state['evaluation']== 'approved' or state['iteration'] >= state['max_iteration']):
        return 'approved'
    else:
        return 'needs_improvement'

In [78]:
graph = StateGraph(TweetState)

graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()
print(workflow)



In [84]:
initial_state = {
    "topic": "hghngnh",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)
result

{'topic': 'hghngnh',
 'tweet': 'Me time: a sacred hour of relaxation... until my brain becomes an Olympic athlete sprinting through my existential crises. Just trying to binge another season of a show I’ll forget by morning! 🤦\u200d♂️ #WhyDoIThinkSoMuch',
 'evaluation': 'approved',
 'feedback': "This tweet offers a relatable and original take on the struggle of trying to enjoy downtime while being plagued by overthinking. The imagery of a brain becoming an Olympic athlete adds a humorous twist and enhances the humor. It's punchy enough to be engaging, and the inclusion of a popular hashtag increases its potential for virality. The format adheres to the guidelines, staying under 280 characters and avoiding a setup-punchline structure. Overall, it captures a universal experience with wit, making it effective and likely to resonate with many.",
 'tweet_history': ['When you finally get some "me time" to relax, but your brain starts running a marathon of existential crises. #hghngnh - it\'s